## Using Tools In LLMs

In [49]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [50]:
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [51]:
system_message="You are a helpful assistant for an Airline called FlightAI."
system_message+="Give short, courteous answers, no more than 1 sentence."
system_message+="Always be accurate. If you don't know the answer, say so."

In [52]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="llama3.2", messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7898

To create a public link, set `share=True` in `launch()`.


## Tools

In [76]:
status = {"london": "active", "paris": "active", "tokyo": "not active", "berlin": "active"}

def get_status(destination_city):
    print(f"Tool get_status called for {destination_city}")
    city = destination_city.lower()
    return status.get(city, "Unknown")

In [78]:
get_status("Berlin")

Tool get_status called for Berlin


'active'

In [80]:
get_status("london")

Tool get_status called for london


'active'

In [57]:
# There's a particular dictionary structure that's required to describe our function:

status_function = {
    "name": "get_status",
    "description": "Get the status of a flight for the distination cities. Call this whenever you need to know the status of the flight, for example when a customer asks 'what is the status of a specific city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [58]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": status_function}]

### Allowing openAI to access this TOOL

In [60]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="llama3.2", messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response, city = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model="llama3.2", messages=messages)
    
    return response.choices[0].message.content

In [61]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    city = arguments.get('destination_city')
    status = get_status(city)
    response = {
        "role": "tool",
        "content": json.dumps({"destination_city": city,"status": status}),
        "tool_call_id": tool_call.id
    }
    return response, city

In [62]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7899

To create a public link, set `share=True` in `launch()`.
